In [1]:
import pandas as pd
import numpy as np

In [2]:
train=pd.read_csv("/content/train.csv")
test=pd.read_csv("/content/test.csv")
train.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [3]:
print(test.shape)

(92757, 12)


In [4]:
train.size

584948

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44996 entries, 0 to 44995
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    44996 non-null  int64  
 1   annual_income         44996 non-null  float64
 2   debt_to_income_ratio  44996 non-null  float64
 3   credit_score          44996 non-null  int64  
 4   loan_amount           44996 non-null  float64
 5   interest_rate         44995 non-null  float64
 6   gender                44995 non-null  object 
 7   marital_status        44995 non-null  object 
 8   education_level       44995 non-null  object 
 9   employment_status     44995 non-null  object 
 10  loan_purpose          44995 non-null  object 
 11  grade_subgrade        44995 non-null  object 
 12  loan_paid_back        44995 non-null  float64
dtypes: float64(5), int64(2), object(6)
memory usage: 4.5+ MB


In [6]:
train.describe()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,loan_paid_back
count,44996.000000,44996.000000,44996.000000,44996.000000,44996.000000,44995.000000,44995.000000
mean,22497.500000,48158.313796,0.120925,680.691528,15001.925263,12.365824,0.796488
std,12989.370693,26848.912948,0.068529,55.287362,6862.393613,2.012589,0.402614
min,0.000000,6093.550000,0.011000,431.000000,500.910000,3.890000,0.000000
25%,11248.750000,27759.960000,0.072000,646.000000,10291.330000,11.000000,1.000000
50%,22497.500000,46463.340000,0.096000,683.000000,14989.630000,12.360000,1.000000
75%,33746.250000,61118.150000,0.156000,719.000000,18791.200000,13.690000,1.000000
max,44995.000000,380653.940000,0.566000,849.000000,48959.950000,20.840000,1.000000


In [7]:

train.columns

Index(['id', 'annual_income', 'debt_to_income_ratio', 'credit_score',
       'loan_amount', 'interest_rate', 'gender', 'marital_status',
       'education_level', 'employment_status', 'loan_purpose',
       'grade_subgrade', 'loan_paid_back'],
      dtype='object')

In [8]:

numeric_cols = train.select_dtypes(include=['number']).columns
train[numeric_cols] = train[numeric_cols].fillna(train[numeric_cols].median())

categorical_cols = train.select_dtypes(exclude=['number']).columns
for col in categorical_cols:
    mode_val = train[col].mode()[0]
    train[col] = train[col].fillna(mode_val)

In [9]:

X = train.drop(['id', 'loan_paid_back'], axis=1)
y = train['loan_paid_back']


X = pd.get_dummies(X, drop_first=True)

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

print("Model trained successfully without any errors!")

Model trained successfully without any errors!


In [13]:
from sklearn.metrics import accuracy_score


y_pred = lr.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.9005555555555556


In [14]:

numeric_cols_test = test.select_dtypes(include=['number']).columns
test[numeric_cols_test] = test[numeric_cols_test].fillna(test[numeric_cols_test].median())

categorical_cols_test = test.select_dtypes(exclude=['number']).columns
for col in categorical_cols_test:
    mode_val = test[col].mode()[0]
    test[col] = test[col].fillna(mode_val)


X_final_test = test.drop(['id'], axis=1)


X_final_test = pd.get_dummies(X_final_test, drop_first=True)

X_final_test = X_final_test.reindex(columns = X.columns, fill_value=0)

X_final_test_scaled = scaler.transform(X_final_test)

final_predictions = lr.predict(X_final_test_scaled)


submission = pd.DataFrame({
    'id': test['id'],
    'loan_paid_back': final_predictions
})

submission.to_csv("submission.csv", index=False)
print("Perfect! New submission.csv created. Shape:", submission.shape)

Perfect! New submission.csv created. Shape: (92757, 2)
